**German Credit Dataset**

# 03.2 - Fairness Models Pipeline

**Objectives**
- Build and run fairness-aware machine learning pipelines with different configurations
- Train and evaluate AIF360 in-processing fairness models automatically
- Export performance and fairness results for all model configurations

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.base import clone
import tensorflow.compat.v1 as tf
import matplotlib

import sys
from pathlib import Path
import importlib
import warnings
from itertools import product
import itertools
import time
from sklearn.preprocessing import StandardScaler

from aif360.datasets import BinaryLabelDataset

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.inprocessing import AdversarialDebiasing, PrejudiceRemover

In [ ]:
sys.path.append('utils')

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

tf.compat.v1.disable_eager_execution()

In [ ]:
import preprocessing
import evaluation

importlib.reload(preprocessing)
importlib.reload(evaluation)

## 1. Load Data

In [ ]:
file_path = '../data/processed/german_df_processed_2.csv'
df = pd.read_csv(file_path, sep=r',', header=0)

In [ ]:
df

## 2. Setup: Key Variables, Helper Functions, and Pipeline Construction

### 2.1. Key Variables

In [ ]:
NUMERIC_COLS = [
    'duration_months', 'credit_amount', 'installment_rate', 
    'residence_duration', 'existing_credits_count', 'dependents'
]

CATEGORICAL_COLS = [
    'sex','age_cat','checking_account_status', 'credit_history', 
    'purpose', 'savings_account_status', 'employment_status', 
    'guarantors', 'property', 'other_debts', 'housing', 'job', 
    'own_telephone?', 'foreign_worker?'
]

In [ ]:
TARGET_COLUMN = 'good_client?'
FAVORABLE_LABEL = 1   # 'Good client'
UNFAVORABLE_LABEL = 0 # 'No good client'

SENSITIVE_ATTR = 'sex'
PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Male
UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Female

# SENSITIVE_ATTR = 'foreign_worker?'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Foreign
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 0}] # Local

# SENSITIVE_ATTR = 'age_cat'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Between 24.0 and 49.1 years old
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Other ages

### 2.2. Helper Functions

In [ ]:
def df_to_aif360(df):
    return BinaryLabelDataset(
        df=df,
        label_names=[TARGET_COLUMN],
        protected_attribute_names=[SENSITIVE_ATTR],
        favorable_label=FAVORABLE_LABEL,
        unfavorable_label=UNFAVORABLE_LABEL
    )

In [ ]:
# to be substitute for preprocessing.py in the future
def encode_features_label_style(ds_train, ds_test):
    """
    Applies ordinal (label-style) encoding to categorical features and standardization scaling.

    This step is required for AIF360 in-processing models that expect fully numeric inputs.
    In particular, scaling helps stabilize optimization and makes the effect of parameters
    such as 'eta' (e.g., in PrejudiceRemover) more noticeable during training.

      1) Extracts feature matrices from AIF360 datasets
      2) Applies OrdinalEncoder to ensure all features are numeric
      3) Applies StandardScaler to normalize features
      4) Copies the original datasets and replaces their features with the processed arrays

    Args:
        ds_train (BinaryLabelDataset): AIF360 training dataset.
        ds_test (BinaryLabelDataset): AIF360 test/validation dataset.

    Returns:
        tuple[BinaryLabelDataset, BinaryLabelDataset]:
            - ds_train_enc: Training dataset with encoded + scaled features.
            - ds_test_enc : Test dataset with encoded + scaled features.
    """
    # ------------------------------------------------------------------
    # 1) Extract raw feature matrices
    # ------------------------------------------------------------------
    X_train = ds_train.features
    X_test  = ds_test.features

    # ------------------------------------------------------------------
    # 2) Encode categoricals into numeric values (ordinal/label style)
    # ------------------------------------------------------------------
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X_train_num = enc.fit_transform(X_train)
    X_test_num  = enc.transform(X_test)

    # ------------------------------------------------------------------
    # 3) Standardize features to stabilize optimization
    # ------------------------------------------------------------------
    scaler = StandardScaler()
    X_train_final = scaler.fit_transform(X_train_num)
    X_test_final  = scaler.transform(X_test_num)

    # ------------------------------------------------------------------
    # 4) Copy datasets and replace features with processed arrays
    # ------------------------------------------------------------------
    ds_train_enc = ds_train.copy()
    ds_test_enc = ds_test.copy()

    ds_train_enc.features = X_train_final
    ds_test_enc.features = X_test_final

    return ds_train_enc, ds_test_enc


In [ ]:
def summarize_folds_to_df_summary(config_id, n_splits, df_folds):
    """
    Aggregates per-fold metrics into a single summary row (mean/std), matching the df_summary format.

      1) Selects metric columns (excluding metadata columns such as config_id/fold/Pipeline)
      2) Computes the mean of each metric across folds (mean_*)
      3) Computes the standard deviation of each metric across folds (std_*)
      4) Builds a single-row DataFrame with config metadata + aggregated metrics

    Args:
        config_id (str): Text identifier of the configuration/model being evaluated.
        n_splits (int): Number of CV folds.
        df_folds (pd.DataFrame): DataFrame containing one row per fold with computed metrics.

    Returns:
        pd.DataFrame: Single-row DataFrame with aggregated metrics:
            - config_id, n_splits
            - mean_* columns for each metric
            - std_* columns for each metric
    """
    # ------------------------------------------------------------------
    # 1) Select metric columns (exclude non-metric metadata)
    # ------------------------------------------------------------------
    metric_cols = [c for c in df_folds.columns if c not in ["config_id", "fold", "Pipeline"]]

    # ------------------------------------------------------------------
    # 2) Compute fold aggregation (mean/std)
    # ------------------------------------------------------------------
    df_mean = df_folds[metric_cols].mean(numeric_only=True).to_frame().T
    df_std  = df_folds[metric_cols].std(numeric_only=True).to_frame().T

    # ------------------------------------------------------------------
    # 3) Build summary DataFrame in the expected format
    # ------------------------------------------------------------------
    df_summary = df_mean.add_prefix("mean_").join(df_std.add_prefix("std_"))
    df_summary.insert(0, "config_id", config_id)
    df_summary.insert(1, "n_splits", n_splits)

    return df_summary


### 2.3. Pipeline Construction

In [ ]:
def run_cv_aif360_inprocessing(model_name, df, n_splits=5, random_state=42, shuffle=True, **model_kwargs):
    """
    Executes Cross-Validation (StratifiedKFold) for a single AIF360 in-processing model and aggregates metrics.

      1) Splits the full DataFrame 'df' into K stratified folds by TARGET_COLUMN
      2) For each fold:
           - Converts df_train/df_test to BinaryLabelDataset (AIF360)
           - Encodes + scales features to numeric arrays (required for in-processing models)
           - Trains the selected in-processing model
           - Predicts on the fold test set
           - Evaluates performance and fairness metrics
      3) Aggregates fold metrics into one summary row (mean/std)

    Args:
        model_name (str): In-processing model name. Supported:
            - "adversarial_debiasing"
            - "prejudice_remover"
        df (pd.DataFrame): Full dataset including TARGET_COLUMN and SENSITIVE_ATTR.
        n_splits (int): Number of CV folds (K).
        random_state (int): Seed (only affects results if shuffle=True).
        shuffle (bool): If True, shuffles the data before creating folds.
        model_kwargs: Model-specific keyword arguments (e.g., eta=... for PrejudiceRemover).

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]:
            - df_folds: DataFrame with one row per fold containing the computed metrics.
            - df_summary: Single-row DataFrame with aggregated metrics (mean_*/std_*).
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    y = df[TARGET_COLUMN].values

    config_id = f"AIF360|model={model_name}|encoder=ordinal|scaler=standardization|{model_kwargs}".replace(" ", "")
    print(f"Running CV for: {config_id} | n_splits={n_splits}")

    fold_rows = []

    # ------------------------------------------------------------------
    # 1) Execute each CV fold
    # ------------------------------------------------------------------
    for fold, (train_idx, test_idx) in enumerate(skf.split(df, y), start=1):
        print(f"\n[Fold {fold}/{n_splits}] Training and evaluating...")

        df_train = df.iloc[train_idx].copy()
        df_test  = df.iloc[test_idx].copy()

        # ------------------------------------------------------------------
        #  1) Convert to AIF360 datasets
        # ------------------------------------------------------------------
        ds_train = df_to_aif360(df_train)
        ds_test  = df_to_aif360(df_test)

        # ------------------------------------------------------------------
        #  2) Encode + scale features (required for in-processing models)
        # ------------------------------------------------------------------
        ds_train_enc, ds_test_enc = encode_features_label_style(ds_train, ds_test)

        # ------------------------------------------------------------------
        #  3) Train + Predict
        # ------------------------------------------------------------------
        if model_name == "prejudice_remover":
            model = PrejudiceRemover(sensitive_attr=SENSITIVE_ATTR, **model_kwargs)
            model.fit(ds_train_enc)
            ds_pred = model.predict(ds_test_enc)

        elif model_name == "adversarial_debiasing":
            tf.reset_default_graph()
            sess = tf.Session()

            model = AdversarialDebiasing(
                privileged_groups=PRIVILEGED_GROUPS,
                unprivileged_groups=UNPRIVILEGED_GROUPS,
                scope_name=f"adv_debias_fold_{fold}",
                sess=sess,
                **model_kwargs
            )
            model.fit(ds_train_enc)
            ds_pred = model.predict(ds_test_enc)

            sess.close()

        else:
            raise ValueError("model_name must be 'prejudice_remover' or 'adversarial_debiasing'")

        # ------------------------------------------------------------------
        #  4) Evaluate performance and fairness, and store fold metrics
        # ------------------------------------------------------------------
        df_metrics = evaluation.evaluate_pipeline(
            ds_test_enc,
            ds_pred,
            UNPRIVILEGED_GROUPS,
            PRIVILEGED_GROUPS,
            pipeline_name=f"{config_id}|fold={fold}"
        )

        df_metrics = df_metrics.copy()
        df_metrics.insert(0, "fold", fold)
        df_metrics.insert(0, "config_id", config_id)
        fold_rows.append(df_metrics)

    # ------------------------------------------------------------------
    # 2) Aggregate fold results into df_folds and df_summary
    # ------------------------------------------------------------------
    df_folds = pd.concat(fold_rows, ignore_index=True)
    df_summary = summarize_folds_to_df_summary(config_id, n_splits, df_folds)

    return df_folds, df_summary

## 3. Pipeline Execution

### 3.1. Single Model Test

In [ ]:
folds_pr, summary_pr = run_cv_aif360_inprocessing(
    "adversarial_debiasing",
    df=df,
    n_splits=3,
    num_epochs=10, 
    classifier_num_hidden_units=100, 
    debias=True,
    adversary_loss_weight=0.1
)

display(summary_pr)

In [ ]:
folds_pr, summary_pr = run_cv_aif360_inprocessing(
    "prejudice_remover",
    df=df,
    n_splits=3,
    eta=25.0
)

display(summary_pr)

### 3.2. Hyperparameter Testing

In [72]:
# ------------------------------------------------------------
# Hyperparameter Search (Model Variants)
# ------------------------------------------------------------

# Parameter grids
etas_to_test = [0.0, 1.0, 5.0, 10.0, 25.0, 50.0, 100.0]

adv_param_grid = {
    "adversary_loss_weight": [0.01, 0.1, 0.5],
    "num_epochs": [50, 150, 300],
    "classifier_num_hidden_units": [64, 128, 200],
    "debias": [True],
}

def run_param_sweep(model_name, param_list, df, n_splits=3):
    """
    Runs run_cv_aif360_inprocessing for multiple parameter settings and returns a compiled results DataFrame.
    """
    summaries = []
    errors = []

    for i, params in enumerate(param_list, start=1):
        print(f"\n[{i}/{len(param_list)}] Running {model_name} with params={params}")

        try:
            folds_df, summary_df = run_cv_aif360_inprocessing(
                model_name,
                df=df,
                n_splits=n_splits,
                **params
            )

            # attach params as columns (so you can filter/sort later)
            for k, v in params.items():
                summary_df[k] = v

            summary_df["model_name"] = model_name
            summaries.append(summary_df)

        except Exception as e:
            errors.append({"model_name": model_name, "params": str(params), "error": str(e)})

    results_df = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
    errors_df = pd.DataFrame(errors)
    return results_df, errors_df

#### 3.2.1 Prejudice Remover

In [73]:
pr_param_list = [{"eta": eta} for eta in etas_to_test]

results_pr, errors_pr = run_param_sweep(
    model_name="prejudice_remover",
    param_list=pr_param_list,
    df=df,
    n_splits=3
)

display(results_pr.sort_values("mean_F1-Score", ascending=False).head(10))
display(errors_pr)


[1/7] Running prejudice_remover with params={'eta': 0.0}
Running CV for: AIF360|model=prejudice_remover|encoder=ordinal|scaler=standardization|{'eta':0.0} | n_splits=3

[Fold 1/3] Training and evaluating...

 RESULTADOS: AIF360|MODEL=PREJUDICE_REMOVER|ENCODER=ORDINAL|SCALER=STANDARDIZATION|{'ETA':0.0}|FOLD=1

--- Métricas de Performance ---
Accuracy:            0.7455
Precision:           0.7968
Recall:              0.8547
F1-Score:            0.8247

--- Métricas de Fairness (Ideal próximo a 0.0) ---
Demographic Parity Diff.:      -0.0735
Equal Opportunity Diff.:       -0.0063
Predictive Parity Diff.:       -0.0946
Average Predictive Value Diff.: 0.0385
Average Odds Diff.:            -0.0323


[Fold 2/3] Training and evaluating...

 RESULTADOS: AIF360|MODEL=PREJUDICE_REMOVER|ENCODER=ORDINAL|SCALER=STANDARDIZATION|{'ETA':0.0}|FOLD=2

--- Métricas de Performance ---
Accuracy:            0.7207
Precision:           0.7692
Recall:              0.8584
F1-Score:            0.8114

--- Métr

,config_id,n_splits,mean_Accuracy,mean_Precision,mean_Recall,mean_F1-Score,mean_Demographic Parity Diff.,mean_Equal Opportunity Diff.,mean_Predictive Parity Diff.,mean_Average Predictive Value Diff.,...,std_Precision,std_Recall,std_F1-Score,std_Demographic Parity Diff.,std_Equal Opportunity Diff.,std_Predictive Parity Diff.,std_Average Predictive Value Diff.,std_Average Odds Diff.,eta,model_name
1,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.735990,0.788137,0.852848,0.819024,-0.098131,-0.064394,-0.044202,-0.003674,...,0.022297,0.010091,0.009526,0.080360,0.077178,0.063662,0.038035,0.108365,1.0,prejudice_remover
0,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.736991,0.791459,0.848563,0.818799,-0.111312,-0.071883,-0.035915,0.001016,...,0.020094,0.013930,0.006816,0.095466,0.089460,0.075223,0.033293,0.128607,0.0,prejudice_remover
5,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.730991,0.775425,0.867142,0.818586,0.009760,0.026526,-0.070790,0.012624,...,0.011000,0.017171,0.005212,0.089233,0.043807,0.067892,0.074908,0.119668,50.0,prejudice_remover
3,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.730985,0.776223,0.865706,0.818321,-0.033033,-0.004744,-0.057243,0.012766,...,0.014327,0.022098,0.008866,0.057818,0.041085,0.066824,0.082680,0.089794,10.0,prejudice_remover
6,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.729990,0.774486,0.867142,0.818044,0.008335,0.026526,-0.069481,0.015107,...,0.012599,0.017171,0.005218,0.090616,0.043807,0.070074,0.075261,0.122475,100.0,prejudice_remover
4,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.729987,0.774390,0.867136,0.818010,-0.014761,-0.007100,-0.076398,-0.020042,...,0.010876,0.019730,0.008489,0.077227,0.052521,0.066276,0.096653,0.108993,25.0,prejudice_remover
2,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.726987,0.776029,0.858571,0.814993,-0.057595,-0.031001,-0.056312,0.001368,...,0.020938,0.012880,0.007492,0.077985,0.067516,0.059026,0.048685,0.103240,5.0,prejudice_remover


""


#### 3.2.2 Adversarial Debiasing

In [74]:
from itertools import product

keys, values = zip(*adv_param_grid.items())
adv_param_list = [dict(zip(keys, v)) for v in product(*values)]

results_adv, errors_adv = run_param_sweep(
    model_name="adversarial_debiasing",
    param_list=adv_param_list,
    df=df,
    n_splits=3
)

display(results_adv.sort_values("mean_F1-Score", ascending=False).head(10))
display(errors_adv)


[1/27] Running adversarial_debiasing with params={'adversary_loss_weight': 0.01, 'num_epochs': 50, 'classifier_num_hidden_units': 64, 'debias': True}
Running CV for: AIF360|model=adversarial_debiasing|encoder=ordinal|scaler=standardization|{'adversary_loss_weight':0.01,'num_epochs':50,'classifier_num_hidden_units':64,'debias':True} | n_splits=3

[Fold 1/3] Training and evaluating...
epoch 0; iter: 0; batch classifier loss: 0.854049; batch adversarial loss: 0.939464
epoch 1; iter: 0; batch classifier loss: 0.847876; batch adversarial loss: 0.938146
epoch 2; iter: 0; batch classifier loss: 0.784741; batch adversarial loss: 0.905373
epoch 3; iter: 0; batch classifier loss: 0.811042; batch adversarial loss: 0.923064
epoch 4; iter: 0; batch classifier loss: 0.803921; batch adversarial loss: 0.909885
epoch 5; iter: 0; batch classifier loss: 0.803174; batch adversarial loss: 0.937353
epoch 6; iter: 0; batch classifier loss: 0.745936; batch adversarial loss: 0.925747
epoch 7; iter: 0; batch c

,config_id,n_splits,mean_Accuracy,mean_Precision,mean_Recall,mean_F1-Score,mean_Demographic Parity Diff.,mean_Equal Opportunity Diff.,mean_Predictive Parity Diff.,mean_Average Predictive Value Diff.,...,std_Demographic Parity Diff.,std_Equal Opportunity Diff.,std_Predictive Parity Diff.,std_Average Predictive Value Diff.,std_Average Odds Diff.,adversary_loss_weight,num_epochs,classifier_num_hidden_units,debias,model_name
19,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.716006,0.762946,0.883056,0.811474,-0.029834,0.036219,-0.018206,0.065556,...,0.049543,0.073401,0.062069,0.086282,0.080387,0.50,50,128,True,adversarial_debiasing
10,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.709008,0.756697,0.905579,0.811049,-0.130631,-0.121832,-0.012518,-0.022555,...,0.226259,0.211020,0.134035,0.044088,0.209658,0.10,50,128,True,adversarial_debiasing
11,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.697979,0.791791,0.806977,0.777739,-0.127285,-0.186211,-0.049414,-0.042968,...,0.357178,0.410223,0.144056,0.143460,0.337301,0.10,50,200,True,adversarial_debiasing
3,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.692099,0.769319,0.813231,0.775942,-0.167517,-0.221386,-0.327449,-0.163051,...,0.449761,0.450345,0.419190,0.217649,0.449702,0.01,150,64,True,adversarial_debiasing
5,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.688008,0.801280,0.744183,0.769124,-0.210868,-0.198639,-0.007340,0.030351,...,0.345957,0.390876,0.103487,0.130527,0.338632,0.01,150,200,True,adversarial_debiasing
8,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.696064,0.814681,0.733031,0.768746,-0.216467,-0.195773,0.015111,0.023566,...,0.250712,0.254881,0.049478,0.054500,0.236412,0.01,300,200,True,adversarial_debiasing
7,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.681007,0.806957,0.717123,0.755868,-0.211992,-0.203926,0.016865,0.031206,...,0.359174,0.432322,0.145540,0.045221,0.339530,0.01,300,128,True,adversarial_debiasing
2,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.666990,0.810604,0.705770,0.739099,-0.407855,-0.431125,-0.268588,-0.172414,...,0.270280,0.341274,0.508280,0.305037,0.216234,0.01,50,200,True,adversarial_debiasing
4,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.666037,0.812244,0.680105,0.735674,-0.355577,-0.394862,-0.011516,-0.031286,...,0.356094,0.361690,0.106131,0.030164,0.361562,0.01,150,128,True,adversarial_debiasing
14,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.668117,0.823241,0.683229,0.725982,-0.157489,-0.210052,-0.012119,-0.016273,...,0.285256,0.279962,0.139521,0.092761,0.288814,0.10,150,200,True,adversarial_debiasing


""


## 3. Exporting Results

In [76]:
results = pd.concat([results_pr, results_adv], ignore_index=True)

display(results.sort_values("mean_F1-Score", ascending=False).head(20))

,config_id,n_splits,mean_Accuracy,mean_Precision,mean_Recall,mean_F1-Score,mean_Demographic Parity Diff.,mean_Equal Opportunity Diff.,mean_Predictive Parity Diff.,mean_Average Predictive Value Diff.,...,std_Equal Opportunity Diff.,std_Predictive Parity Diff.,std_Average Predictive Value Diff.,std_Average Odds Diff.,eta,model_name,adversary_loss_weight,num_epochs,classifier_num_hidden_units,debias
1,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.735990,0.788137,0.852848,0.819024,-0.098131,-0.064394,-0.044202,-0.003674,...,0.077178,0.063662,0.038035,0.108365,1.0,prejudice_remover,NaN,NaN,NaN,NaN
0,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.736991,0.791459,0.848563,0.818799,-0.111312,-0.071883,-0.035915,0.001016,...,0.089460,0.075223,0.033293,0.128607,0.0,prejudice_remover,NaN,NaN,NaN,NaN
5,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.730991,0.775425,0.867142,0.818586,0.009760,0.026526,-0.070790,0.012624,...,0.043807,0.067892,0.074908,0.119668,50.0,prejudice_remover,NaN,NaN,NaN,NaN
3,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.730985,0.776223,0.865706,0.818321,-0.033033,-0.004744,-0.057243,0.012766,...,0.041085,0.066824,0.082680,0.089794,10.0,prejudice_remover,NaN,NaN,NaN,NaN
6,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.729990,0.774486,0.867142,0.818044,0.008335,0.026526,-0.069481,0.015107,...,0.043807,0.070074,0.075261,0.122475,100.0,prejudice_remover,NaN,NaN,NaN,NaN
4,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.729987,0.774390,0.867136,0.818010,-0.014761,-0.007100,-0.076398,-0.020042,...,0.052521,0.066276,0.096653,0.108993,25.0,prejudice_remover,NaN,NaN,NaN,NaN
2,AIF360|model=prejudice_remover|encoder=ordinal...,3,0.726987,0.776029,0.858571,0.814993,-0.057595,-0.031001,-0.056312,0.001368,...,0.067516,0.059026,0.048685,0.103240,5.0,prejudice_remover,NaN,NaN,NaN,NaN
26,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.716006,0.762946,0.883056,0.811474,-0.029834,0.036219,-0.018206,0.065556,...,0.073401,0.062069,0.086282,0.080387,NaN,adversarial_debiasing,0.50,50.0,128.0,True
17,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.709008,0.756697,0.905579,0.811049,-0.130631,-0.121832,-0.012518,-0.022555,...,0.211020,0.134035,0.044088,0.209658,NaN,adversarial_debiasing,0.10,50.0,128.0,True
18,AIF360|model=adversarial_debiasing|encoder=ord...,3,0.697979,0.791791,0.806977,0.777739,-0.127285,-0.186211,-0.049414,-0.042968,...,0.410223,0.144056,0.143460,0.337301,NaN,adversarial_debiasing,0.10,50.0,200.0,True


In [77]:
file_out_path = '../data/results'

results.to_csv(file_out_path + '/fairness_models_results.csv', index=False)